#### MFVI -- Mean-Field Variational Inference (Bayes-by-Backprop) auf Qwen-LoRA

Nach Blundell et al. 2015 ("Weight Uncertainty in Neural Networks"). Baut auf `qwen_posterior_utils.py` auf (gleicher `theta_map`, gleiches 128er-Posterior-Subset, gleiche Auswertungspipeline wie in `02_methods_v3.ipynb` fuer MILE) -- fuer einen fairen Methodenvergleich.


In [1]:
import os
os.environ["N_PER_CLASS"] = "2048"
os.environ["SEED"] = "1"

In [2]:
from pathlib import Path

# ============================================================
# CONFIG
# MFVI
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO
    / "experiments"
    / "ag_news_qwen_lora"
    / "evaluate_saved_baseline.py"
)

RESULT_DIR = (
    MASTER_DIR
    / "method_results"
    / "mfvi_qwen_agnews"
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

# Standardwerte greifen beim interaktiven Ausfuehren wie bisher.
# Fuer SLURM-Batch-Jobs koennen SEED/N_PER_CLASS/N_SAMPLES per
# Umgebungsvariable ueberschrieben werden, ohne das Notebook zu duplizieren.
import os

SEED = int(os.environ.get("SEED", "1"))   # Zusatzlauf zur Multimodalitaets-Kontrolle (Seed 2 bleibt der Hauptlauf)
POSTERIOR_KEY_SEED = 2027   # gleich wie MILE -- gleiches Subset

# ------------------------------------------------------------
# Posterior data (identisch zu 02_methods_v3.ipynb / MILE)
# ------------------------------------------------------------

N_PER_CLASS = int(os.environ.get("N_PER_CLASS", "512"))        # erweitert von 128 -- 512 x 4 = 2048 examples
SEQ_LEN = 32

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS  # vorgezogen, wird unten fuer
                                       # KL_WEIGHT_MAX gebraucht

# ------------------------------------------------------------
# Prior (identisch zu MILE)
# ------------------------------------------------------------

# War 1.0 (Standard-Default aus dem MILE-Paper). Bei 540672 Dimensionen
# ist das massiv zu breit: eine typische Stichprobe aus N(0, 1.0^2 * I)
# hat eine Norm von ~sqrt(540672)*1.0 ~= 735 -- weit weg von der
# tatsaechlichen MAP-Norm (~13.0). Das MILE-Paper selbst skaliert die
# Prior-Varianz fuer groessere Modelle runter (0.1-0.4 statt 1.0 fuer
# ihre CNN/ATT-Modelle, deutlich kleiner als unser 540k-dim LoRA-Raum).
# Empirisch hergeleitet aus der MAP-Norm: 13.0148 / sqrt(540672) ~= 0.0177.
PRIOR_STD = 0.02

# ------------------------------------------------------------
# MFVI
# ------------------------------------------------------------

N_ELBO_STEPS = int(os.environ.get("N_ELBO_STEPS", "200"))
KL_WARMUP_STEPS = 150   # linearer KL-Weight-Anstieg 0->1 ueber diese Schritte
# KL_WEIGHT_MAX = 1/N (N = Groesse des Posterior-Subsets) -- Standard-
# Downweighting-Heuristik bei Mean-Field-VI mit kleinem Datensatz und
# riesiger Parameterzahl (540672). Ersetzt den vorherigen festen Wert 0.3:
# bei 0.3 machte die KL im letzten Trainingslauf 99.8% des optimierten
# Loss aus (0.3*249022 vs. nll=139 im finalen Schritt) -- das Datensignal
# ging darin praktisch unter (Testset-Accuracy fiel auf 46.1%). Mit
# 1/N_POSTERIOR_CONFIG ergibt sich ein deutlich moderateres Verhaeltnis
# (~3.5:1 statt ~536:1 KL zu NLL, hochgerechnet aus denselben Werten).
KL_WEIGHT_MAX = 1.0 / N_POSTERIOR_CONFIG
GRAD_CLIP_NORM = 1.0    # globale Gradientennorm-Kappung -- kl (~560k) ist so
                        # viel groesser als nll (~34), dass selbst kleine
                        # kl_weight-Werte riesige Gradienten erzeugen koennen
LEARNING_RATE = 1e-3

# sigma_init = softplus(INIT_RHO) -- klein, damit das Training nahe
# der schon trainierten MAP-Loesung startet.
INIT_RHO = -5.0

SIGMA_FLOOR = 1e-6   # numerische Untergrenze fuer sigma

N_SAMPLES = int(os.environ.get("N_SAMPLES", "10"))   # Posterior-Draws fuer die Auswertung (wie bei MILE)

PROGRESS_EVERY = 20

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64

# False (Default)  -> auswerten auf dem 128er-Posterior-Subset
#                      (schneller Dev-/Stabilitaets-Check).
# True              -> auswerten auf dem echten AG-News-Testset
#                      (7600 Beispiele, fuer die finalen Thesis-Zahlen).
# EVAL_MODE steuert die Auswertungsdaten -- WICHTIG: fuer Hyperparameter-
# Tuning (z.B. KL_WEIGHT_MAX-Kandidaten testen) IMMER "val" nutzen, nie
# "test". Sonst tunen wir de facto auf dem Testset und die finalen Zahlen
# waeren nicht mehr ehrlich "auf ungesehenen Daten".
#   "posterior" -> dieselben 512 Beispiele wie beim Fitting (Sanity-Check)
#   "val"       -> 12.000 Validation-Beispiele (Hyperparameter-Tuning)
#   "test"      -> 7.600 echte Testbeispiele (NUR fuer den finalen,
#                  einmaligen Report, erst wenn EVAL_MODE="val" gut aussieht)
EVAL_MODE = "test"

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

RUN_NAME = (
    f"mfvi_seed{SEED}"
    f"_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_steps{N_ELBO_STEPS}"
    f"_samples{N_SAMPLES}"
    f"_{EVAL_MODE}eval"
)

print("======================================")
print("MFVI CONFIG")
print("======================================")
print("Run name:          ", RUN_NAME)
print("Seed:              ", SEED)
print("Examples/class:    ", N_PER_CLASS)
print("Total examples:    ", N_POSTERIOR_CONFIG)
print("Sequence length:   ", SEQ_LEN)
print("Prior std:         ", PRIOR_STD)
print("ELBO steps:        ", N_ELBO_STEPS)
print("KL warmup steps:   ", KL_WARMUP_STEPS)
print("Learning rate:     ", LEARNING_RATE)
print("Init rho:          ", INIT_RHO)
print("Posterior samples: ", N_SAMPLES)
print("======================================")


MFVI CONFIG
Run name:           mfvi_seed1_balanced8192_seq32_prior0.02_steps200_samples10_testeval
Seed:               1
Examples/class:     2048
Total examples:     8192
Sequence length:    32
Prior std:          0.02
ELBO steps:         200
KL warmup steps:    150
Learning rate:      0.001
Init rho:           -5.0
Posterior samples:  10


In [3]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import time
import runpy

import numpy as np
import jax
import jax.numpy as jnp
import optax

from jax import random


RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()

NOTEBOOK_START_TIME = time.time()  # Gesamtlaufzeit ab hier gemessen


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3
Devices: [CudaDevice(id=0)]

Paths:
Qwen repo: True
Baseline script: True
Result directory: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mfvi_qwen_agnews


In [4]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [5]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

test_metrics = baseline_objects["test_metrics"]

print("\nBaseline loaded")
print("Model:", type(env.s_model))

print("\nBaseline test metrics:")
for key, value in test_metrics.items():
    print(f"{key}: {float(value):.6f}")


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246214
val_acc: 0.913667
val_ece: 0.009068
val_brier: 0.131597
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000

Baseline loaded
Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>

Baseline test metrics:
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000


In [6]:
# ============================================================
# SEED-SPEZIFISCHE PARAMS LADEN (nur noetig, wenn SEED != 2 -- die
# Baseline aus BASELINE_SCRIPT ist fest auf Seed 2 verdrahtet. Gleiche
# Technik wie in 02_mile_multichain.ipynb / 05_deep_ensemble.ipynb:
# parametrisierte Kopie von evaluate_saved_baseline.py mit anderem
# CHECKPOINT_PATH/SEED.)
# ============================================================

CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"


def load_seed_params(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_path = list(seed_dir.glob("*_pretrained_params.npy"))[0]

    eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")

    old_checkpoint_line = next(
        line for line in eval_script_template.splitlines()
        if line.strip().startswith("CHECKPOINT_PATH")
    )
    old_seed_line = next(
        line for line in eval_script_template.splitlines()
        if line.strip().startswith("SEED")
    )

    script_text = eval_script_template.replace(
        old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(
        old_seed_line, f"SEED = {seed}",
    )

    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora"
        / f"_tmp_eval_mfvi_seed{seed}.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        seed_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)

    return seed_objects["params"]


if SEED != 2:
    print(f"Ersetze Baseline-Params (Seed 2) durch Seed {SEED} ...")
    params = load_seed_params(SEED)
    print(f"Params fuer Seed {SEED} geladen.")
else:
    print("SEED == 2 -> nutzt die bereits geladenen Baseline-Params.")


Ersetze Baseline-Params (Seed 2) durch Seed 1 ...


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1/iiljya0g_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.247482
val_acc: 0.912250
val_ece: 0.008029
val_brier: 0.131916
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.252232
test_acc: 0.914474
test_ece: 0.007562
test_brier: 0.132536
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Params fuer Seed 1 geladen.


In [7]:
# ============================================================
# POSTERIOR SETUP (gleiches Subset, gleicher theta_map wie MILE)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

theta_map = posterior.theta_map
rebuild_full_params = posterior.rebuild_full_params
qwen_subset_nll_sum = posterior.qwen_subset_nll_sum
qwen_log_posterior = posterior.qwen_log_posterior


Subset indices: [   0    1    4 ... 8128 8131 8135]
Labels: [0 0 0 ... 3 3 3]
Number of examples: 8192
input_ids: (8192, 32)
attention_mask: (8192, 32)
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True


### ELBO

`nll_fn(theta)` ist dieselbe NLL wie in MILEs `qwen_log_posterior`, nur ohne den Prior-Term -- der Prior geht hier stattdessen analytisch in die Gauss-KL ein (niedrigere Varianz als ein Monte-Carlo-Schaetzer des Priors, Standardpraxis bei Bayes-by-Backprop mit Gauss-Prior + Gauss-Posterior).


In [8]:
# ============================================================
# ELBO / REPARAMETRIZATION
# ============================================================

D = int(theta_map.shape[0])


def nll_fn(theta):
    candidate_params = rebuild_full_params(theta)
    nll_sum, _ = qwen_subset_nll_sum(
        candidate_params,
        x_posterior,
        y_posterior,
        posterior_example_keys,
    )
    return nll_sum


def gaussian_kl(mu, sigma, prior_std):
    """KL[N(mu, sigma^2) || N(0, prior_std^2)], summed over all dims."""
    return jnp.sum(
        jnp.log(prior_std / sigma)
        + (jnp.square(sigma) + jnp.square(mu)) / (2.0 * prior_std**2)
        - 0.5
    )


def negative_elbo(variational_params, rng_key, kl_weight):
    """kl_weight in [0, 1] -- KL-Warmup: mit vollem KL-Gewicht ab Schritt 0
    zieht der Prior-Term (~560k) das Optimum sofort weit von theta_map weg
    (NLL steigt von ~34 auf >100 innerhalb von 20 Schritten), weil er den
    winzigen Datenterm (~34) um Groessenordnungen dominiert. kl_weight faehrt
    den KL-Term stattdessen langsam hoch, damit der Datenfit zuerst greifen
    kann, bevor die Prior-Regularisierung einsetzt."""
    mu = variational_params["mu"]
    rho = variational_params["rho"]

    sigma = jax.nn.softplus(rho) + SIGMA_FLOOR

    epsilon = random.normal(rng_key, mu.shape, dtype=mu.dtype)
    theta_sample = mu + sigma * epsilon

    nll = nll_fn(theta_sample)
    kl = gaussian_kl(mu, sigma, PRIOR_STD)

    neg_elbo = nll + kl_weight * kl

    return neg_elbo, {
        "nll": nll,
        "kl": kl,
        "kl_weight": kl_weight,
        "sigma_mean": sigma.mean(),
    }


In [9]:
# ============================================================
# OPTIMIZER + TRAIN_STEP (wiederverwendbar fuer Sweep + finalen Lauf)
# ============================================================

# Gradient-Clipping VOR adamw: begrenzt die globale Norm des Updates,
# unabhaengig davon wie gross der rohe (vom KL-Term dominierte) Gradient
# ist -- verhindert einzelne katastrophale Schritte, die zu NaN fuehren.
optimizer = optax.chain(
    optax.clip_by_global_norm(GRAD_CLIP_NORM),
    optax.adamw(LEARNING_RATE),
)

grad_fn = jax.value_and_grad(negative_elbo, has_aux=True)


# @jax.jit sorgt dafuer, dass train_step nur EINMAL kompiliert wird und
# danach fuer alle Schritte (und alle Sweep-Kandidaten) wiederverwendet
# wird (kl_weight ist ein Funktionsargument, keine Python-Closure-
# Konstante -- aendert sich pro Schritt, ohne dass neu kompiliert wird).
@jax.jit
def train_step(variational_params, opt_state, rng_key, kl_weight):
    (loss, aux), grads = grad_fn(variational_params, rng_key, kl_weight)
    updates, opt_state = optimizer.update(grads, opt_state, variational_params)
    variational_params = optax.apply_updates(variational_params, updates)
    return variational_params, opt_state, loss, aux


def kl_weight_schedule(step, kl_weight_max):
    ramp = min(step / KL_WARMUP_STEPS, 1.0)
    return float(ramp * kl_weight_max)


def run_mfvi_training(kl_weight_max, rng_key, verbose=True):
    """Trainiert eine frische MFVI-Instanz (mu=theta_map, rho=INIT_RHO) fuer
    N_ELBO_STEPS mit dem gegebenen kl_weight_max. Wird sowohl vom Sweep
    (mehrfach, pro Kandidat) als auch vom finalen Trainingslauf (einmal,
    mit dem ausgewaehlten Wert) benutzt -- garantiert identischen
    Trainingscode fuer beide."""
    variational_params = {
        "mu": theta_map,
        "rho": jnp.full((D,), INIT_RHO, dtype=jnp.float32),
    }
    opt_state = optimizer.init(variational_params)

    loss_history = []
    nll_history = []
    kl_history = []
    kl_weight_history = []
    sigma_mean_history = []

    for step in range(N_ELBO_STEPS):
        rng_key, step_key = random.split(rng_key)

        kl_weight = kl_weight_schedule(step, kl_weight_max)

        variational_params, opt_state, loss, aux = train_step(
            variational_params, opt_state, step_key, kl_weight,
        )

        loss_host = float(loss)
        nll_host = float(aux["nll"])
        kl_host = float(aux["kl"])
        sigma_mean_host = float(aux["sigma_mean"])

        loss_history.append(loss_host)
        nll_history.append(nll_host)
        kl_history.append(kl_host)
        kl_weight_history.append(kl_weight)
        sigma_mean_history.append(sigma_mean_host)

        finite = np.isfinite(loss_host)

        if verbose and (
            step % PROGRESS_EVERY == 0 or step == N_ELBO_STEPS - 1 or not finite
        ):
            print(
                f"step {step:4d}  kl_weight={kl_weight:.5f}  neg_elbo={loss_host:12.3f}  "
                f"nll={nll_host:12.3f}  kl={kl_host:10.3f}  "
                f"sigma_mean={sigma_mean_host:.6f}"
            )

        if not finite:
            raise FloatingPointError(
                f"MFVI diverged at step {step} (kl_weight_max={kl_weight_max}): "
                f"neg_elbo is not finite."
            )

    history = {
        "loss_history": loss_history,
        "nll_history": nll_history,
        "kl_history": kl_history,
        "kl_weight_history": kl_weight_history,
        "sigma_mean_history": sigma_mean_history,
    }

    return variational_params, rng_key, history


In [20]:
# ============================================================
# KL_WEIGHT_MAX SWEEP -- waehlt automatisch den besten Kandidaten anhand
# der VAL-SET-Accuracy (NIEMALS auf dem Testset sweepen -- sonst tunen
# wir de facto auf den Testdaten und die finalen Zahlen waeren verzerrt).
# ============================================================

RUN_KL_WEIGHT_SWEEP = False  # aus Zeitgruenden uebersprungen (nur ~1h28 verfuegbar) --
                             # nutzt stattdessen denselben *relativen* Multiplikator
                             # (0.3x), der beim Sweep auf 512 Beispielen gewonnen hat.
                             # Die Basis 1/N_POSTERIOR_CONFIG skaliert automatisch mit
                             # der Datenmenge mit, daher sollte der relative Wert auch
                             # bei 2048 Beispielen ungefaehr passen -- nicht erneut
                             # empirisch bestaetigt, als Limitation zu dokumentieren.
FALLBACK_KL_WEIGHT_MAX = 1 / N_POSTERIOR_CONFIG

BASE_KL_WEIGHT = 1.0 / N_POSTERIOR_CONFIG
KL_WEIGHT_RELATIVE_CANDIDATES = [0.1, 0.3, 1.0, 3.0, 10.0]

if RUN_KL_WEIGHT_SWEEP:
    assert EVAL_MODE == "val", (
        "KL-Weight-Sweep darf nur mit EVAL_MODE='val' laufen -- sonst "
        "tunen wir de facto auf dem Testset."
    )

    x_eval_sweep, y_eval_sweep = data.get("val")
    print("Sweep-Eval-Beispiele (Val-Set):", int(y_eval_sweep.shape[0]))

    sweep_results = []
    sweep_rng_key = rng_key
    sweep_start_time = time.time()

    for relative_multiplier in KL_WEIGHT_RELATIVE_CANDIDATES:
        candidate_start_time = time.time()
        candidate_kl_weight_max = relative_multiplier * BASE_KL_WEIGHT
        print(
            f"\n=== KL_WEIGHT_MAX candidate: {candidate_kl_weight_max:.6f} "
            f"(relativ {relative_multiplier}x von 1/N) ==="
        )

        candidate_variational_params, sweep_rng_key, _ = run_mfvi_training(
            kl_weight_max=candidate_kl_weight_max,
            rng_key=sweep_rng_key,
            verbose=False,
        )

        mu_c = candidate_variational_params["mu"]
        sigma_c = jax.nn.softplus(candidate_variational_params["rho"]) + SIGMA_FLOOR

        sweep_rng_key, sampling_key_c = random.split(sweep_rng_key)
        sampling_keys_c = random.split(sampling_key_c, N_SAMPLES)

        def draw_one_sample_c(key, mu_c=mu_c, sigma_c=sigma_c):
            epsilon = random.normal(key, mu_c.shape, dtype=mu_c.dtype)
            return mu_c + sigma_c * epsilon

        sample_positions_c = jax.vmap(draw_one_sample_c)(sampling_keys_c)
        samples_host_c = np.asarray(jax.device_get(sample_positions_c))

        sample_probabilities_c, mean_probabilities_c, sweep_rng_key = (
            qpu.compute_posterior_predictive_probabilities(
                env=env,
                rebuild_full_params=rebuild_full_params,
                sample_thetas=samples_host_c,
                x_eval=x_eval_sweep,
                y_eval=y_eval_sweep,
                rng_key=sweep_rng_key,
                eval_batch_size=EVAL_BATCH_SIZE,
            )
        )

        metrics_c = qpu.compute_predictive_metrics(
            sample_probabilities=sample_probabilities_c,
            mean_probabilities=mean_probabilities_c,
            y_true=y_eval_sweep,
        )

        candidate_duration = time.time() - candidate_start_time
        print(f"Dauer Kandidat {relative_multiplier}x: {candidate_duration:.1f}s ({candidate_duration/60:.1f} min)")

        sweep_results.append({
            "relative_multiplier": relative_multiplier,
            "kl_weight_max": candidate_kl_weight_max,
            "accuracy": float(metrics_c["accuracy"]),
            "nll": float(metrics_c["posterior_predictive_nll"]),
            "brier": float(metrics_c["brier_score"]),
            "duration_seconds": candidate_duration,
        })

    sweep_total_duration = time.time() - sweep_start_time

    print("\n======================================")
    print("SWEEP SUMMARY (Val-Set)")
    print("======================================")
    for r in sweep_results:
        print(
            f"{r['relative_multiplier']:>6}x  (KL_WEIGHT_MAX={r['kl_weight_max']:.6f})  "
            f"Accuracy={r['accuracy']:.4f}  NLL={r['nll']:.4f}  Brier={r['brier']:.4f}  "
            f"Dauer={r['duration_seconds']:.1f}s"
        )
    print(f"Sweep-Gesamtdauer: {sweep_total_duration:.1f}s ({sweep_total_duration/60:.1f} min)")

    best = max(sweep_results, key=lambda r: r["accuracy"])
    KL_WEIGHT_MAX = best["kl_weight_max"]

    print()
    print(
        f"AUSGEWAEHLT: KL_WEIGHT_MAX = {KL_WEIGHT_MAX:.6f} "
        f"(relativ {best['relative_multiplier']}x von 1/N), "
        f"Val-Accuracy={best['accuracy']:.4f}"
    )

    if best["relative_multiplier"] == max(KL_WEIGHT_RELATIVE_CANDIDATES):
        print()
        print(
            "HINWEIS: der groesste getestete Wert war noch der beste -- "
            "KL_WEIGHT_RELATIVE_CANDIDATES evtl. nach oben erweitern und "
            "neu laufen lassen."
        )
    if best["relative_multiplier"] == min(KL_WEIGHT_RELATIVE_CANDIDATES):
        print()
        print(
            "HINWEIS: der kleinste getestete Wert war noch der beste -- "
            "KL_WEIGHT_RELATIVE_CANDIDATES evtl. nach unten erweitern und "
            "neu laufen lassen."
        )

else:
    KL_WEIGHT_MAX = FALLBACK_KL_WEIGHT_MAX
    print(f"Sweep uebersprungen. KL_WEIGHT_MAX = {KL_WEIGHT_MAX:.6f}")


Sweep uebersprungen. KL_WEIGHT_MAX = 0.000122


In [21]:
# ============================================================
# TRAINING LOOP (finaler Lauf mit dem oben ausgewaehlten KL_WEIGHT_MAX)
# ============================================================

print("======================================")
print("MFVI TRAINING (final)")
print("======================================")
print("KL warmup steps:", KL_WARMUP_STEPS)
print("KL weight max:  ", KL_WEIGHT_MAX)

final_training_start_time = time.time()
variational_params, rng_key, training_history = run_mfvi_training(
    kl_weight_max=KL_WEIGHT_MAX,
    rng_key=rng_key,
    verbose=True,
)
final_training_duration = time.time() - final_training_start_time

loss_history = training_history["loss_history"]
nll_history = training_history["nll_history"]
kl_history = training_history["kl_history"]
kl_weight_history = training_history["kl_weight_history"]
sigma_mean_history = training_history["sigma_mean_history"]

print()
print("MFVI training completed.")
print("Final neg ELBO:", loss_history[-1])
print("Final mean sigma:", sigma_mean_history[-1])
print(f"Trainingsdauer: {final_training_duration:.1f}s ({final_training_duration/60:.1f} min)")


MFVI TRAINING (final)
KL warmup steps: 150
KL weight max:   0.0001220703125
step    0  kl_weight=0.00000  neg_elbo=    2952.640  nll=    2952.640  kl=561778.000  sigma_mean=0.006716
step   20  kl_weight=0.00002  neg_elbo=    2236.827  nll=    2227.302  kl=585225.688  sigma_mean=0.006717
step   40  kl_weight=0.00003  neg_elbo=    1012.993  nll=     992.554  kl=627898.875  sigma_mean=0.006717
step   60  kl_weight=0.00005  neg_elbo=     187.319  nll=     154.151  kl=679282.625  sigma_mean=0.006718
step   80  kl_weight=0.00007  neg_elbo=     129.556  nll=      83.124  kl=713203.000  sigma_mean=0.006720
step  100  kl_weight=0.00008  neg_elbo=      93.750  nll=      34.251  kl=731123.250  sigma_mean=0.006726
step  120  kl_weight=0.00010  neg_elbo=     141.124  nll=      69.231  kl=736182.312  sigma_mean=0.006736
step  140  kl_weight=0.00011  neg_elbo=     191.457  nll=     108.056  kl=732020.938  sigma_mean=0.006751
step  160  kl_weight=0.00012  neg_elbo=     104.397  nll=      15.767  kl=72

FloatingPointError: MFVI diverged at step 196 (kl_weight_max=0.0001220703125): neg_elbo is not finite.

In [ ]:
# ============================================================
# DRAW POSTERIOR SAMPLES FROM TRAINED q
# ============================================================

mu_final = variational_params["mu"]
sigma_final = jax.nn.softplus(variational_params["rho"]) + SIGMA_FLOOR

rng_key, sampling_key = random.split(rng_key)
sampling_keys = random.split(sampling_key, N_SAMPLES)


def draw_one_sample(key):
    epsilon = random.normal(key, mu_final.shape, dtype=mu_final.dtype)
    return mu_final + sigma_final * epsilon


sample_positions = jax.vmap(draw_one_sample)(sampling_keys)
sample_positions = jax.block_until_ready(sample_positions)

samples_host = np.asarray(jax.device_get(sample_positions))

finite_per_sample = np.isfinite(samples_host).all(axis=1)

print("Samples shape:", samples_host.shape)
print("Mean sigma:", float(sigma_final.mean()))
print("Min/Max sigma:", float(sigma_final.min()), float(sigma_final.max()))
print("All samples finite:", finite_per_sample.all())

assert finite_per_sample.all(), "At least one MFVI sample contains NaN or Inf."


In [ ]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS
# ============================================================

distances_from_map = qpu.compute_distances_from_map(theta_map, sample_positions)


In [ ]:
# ============================================================
# EVALUATION DATA (Posterior-Subset / Validation / Testset)
# ============================================================

if EVAL_MODE == "test":
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set (final report).")
elif EVAL_MODE == "val":
    x_eval, y_eval = data.get("val")
    print("Evaluating on the AG News VALIDATION set (hyperparameter tuning).")
elif EVAL_MODE == "posterior":
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")
else:
    raise ValueError(f"Unknown EVAL_MODE: {EVAL_MODE!r}")

print("Evaluation examples:", int(y_eval.shape[0]))


In [ ]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=samples_host,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


In [ ]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


In [ ]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


In [ ]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior,
    sample_positions,
)


In [ ]:
# ============================================================
# SAVE RUN
# ============================================================

notebook_total_duration = time.time() - NOTEBOOK_START_TIME

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

summary.update({
    "run_name": RUN_NAME,
    "method": "MFVI",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "seed": SEED,
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_mode": EVAL_MODE,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,

    "n_elbo_steps": N_ELBO_STEPS,
    "kl_warmup_steps": KL_WARMUP_STEPS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "kl_weight_max": KL_WEIGHT_MAX,
    "learning_rate": LEARNING_RATE,
    "init_rho": INIT_RHO,
    "n_samples": N_SAMPLES,

    "final_neg_elbo": loss_history[-1],
    "final_nll": nll_history[-1],
    "final_kl": kl_history[-1],
    "final_sigma_mean": sigma_mean_history[-1],
    "final_sigma_min": float(sigma_final.min()),
    "final_sigma_max": float(sigma_final.max()),
    "final_training_duration_seconds": final_training_duration,
    "sweep_ran": RUN_KL_WEIGHT_SWEEP,

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),

    "notebook_total_duration_seconds": notebook_total_duration,
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
    "loss_history": np.asarray(loss_history),
    "nll_history": np.asarray(nll_history),
    "kl_history": np.asarray(kl_history),
    "sigma_mean_history": np.asarray(sigma_mean_history),
    "kl_weight_history": np.asarray(kl_weight_history),
    "final_sigma": np.asarray(jax.device_get(sigma_final)),
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
print()
print(f"Trainingsdauer: {final_training_duration:.1f}s ({final_training_duration/60:.1f} min)")
print(f"Gesamtlaufzeit Notebook: {notebook_total_duration:.1f}s ({notebook_total_duration/60:.1f} min)")
